# Bahrain Import Analysis 2021–2025
### Data Analytics Bootcamp — General Assembly

Python is used to clean and consolidate raw import data across five years. Power BI is used for analysis, insight generation, and interactive dashboarding.

**Data Source:** [Bahrain Open Data Portal](https://www.data.gov.bh/explore/?disjunctive.theme&sort=modified&q=imports)

> To run this notebook, download the five import CSV files (2021–2025) from the link above and place them in `/content/`.

---

## Data Dictionary

| Column | Type | Description |
|--------|------|-------------|
| `year` | int | Calendar year of the import record |
| `month` | str | Month name in English (e.g. "January") |
| `commodity_no` | str | 8-digit HS code — kept as string to preserve leading zeros |
| `commodity` | str | English description of the imported commodity |
| `un_code` | str | UN country code for the origin country |
| `country_name` | str | Country from which the goods were imported |
| `import_value_bd` | float | Import value in Bahraini Dinar (BHD) |
| `import_value_usa` | float | Import value in US Dollars (USD) |
| `import_weight_kg` | float | Total weight in kilograms |
| `import_quantity` | float | Quantity in the unit specified by `um` — do not aggregate across different units |
| `um` | str | Unit of measurement for `import_quantity` (e.g. KG, TON, NO) |

> **Note on 2024:** The 2024 file uses Arabic column names and is missing the `year` column. Both are corrected during cleaning. Arabic duplicate columns (`السلعة`, `الدولة`) and the row index column (`N`) are dropped across all files.

---

## Business Questions

1. How have total import volumes and values trended from 2021 to 2025?
2. Which commodities are most imported and what is their price range?
3. Which countries are the most common and most cost-effective sourcing partners?
4. Are there seasonal demand patterns by commodity, month, or year?
5. Are there niche opportunities — underserved markets or unusual import patterns?

---
## 1. Load Raw Data

In [8]:
import pandas as pd
import os

csv_files = [
    '/content/import-2021.csv',
    '/content/import-2022.csv',
    '/content/import-2023.csv',
    '/content/import-2024.csv',
    '/content/import-2025.csv'
]

df_collection = {}

print('Inspecting columns for each file:')
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    try:
        # 'Commodity No' must be read as str to preserve leading zeros
        df = pd.read_csv(file_path, dtype={'Commodity No': str})
        df_collection[file_name] = df
        print(f'\n--- Columns for {file_name} ---')
        print(df.columns.tolist())
    except Exception as e:
        print(f'Error reading {file_name}: {e}')

---
## 2. Preview Raw Files

In [9]:
for file_name in ['import-2021.csv', 'import-2022.csv']:
    print(f'\n--- Head of {file_name} ---')
    display(df_collection[file_name].head())

In [10]:
for file_name in ['import-2023.csv', 'import-2024.csv', 'import-2025.csv']:
    print(f'\n--- Head of {file_name} ---')
    display(df_collection[file_name].head())

---
## 3. Clean & Consolidate

For each file:
- Drop the row-index column (`N`) and Arabic duplicate columns (`السلعة`, `الدولة`)
- Inject `year = 2024` for the 2024 file (column was missing)
- Standardize column names via mapping
- Normalize the `month` column to plain month names (e.g. `'01 January'` → `'January'`)
- Align all files to the same 11-column schema
- Concatenate into a single master DataFrame and remove stray newline characters

In [11]:
cleaned_dataframes = []

column_name_mapping = {
    'Year': 'year',
    'Month': 'month',
    'Commodity No': 'commodity_no',
    'Commodity ': 'commodity',
    'Commodity': 'commodity',
    'UN code ': 'un_code',
    'UN code': 'un_code',
    'Country Name': 'country_name',
    'Import Value (BD)': 'import_value_bd',
    'Import Value (USA $)': 'import_value_usa',
    'Import Weight (KG)': 'import_weight_kg',
    'Import Quantity': 'import_quantity',
    'UM': 'um',
    'قيمة الواردات (دينار بحريني)': 'import_value_bd',
    'قيمة الواردات (دولار أمريكي)': 'import_value_usa',
    'وزن الواردات (كجم)': 'import_weight_kg',
    'كمية الواردات': 'import_quantity',
    'وحدة القياس': 'um'
}

arabic_cols_to_drop = ['السلعة', 'الدولة']

expected_columns = [
    'year', 'month', 'commodity_no', 'commodity', 'un_code',
    'country_name', 'import_value_bd', 'import_value_usa',
    'import_weight_kg', 'import_quantity', 'um'
]

for file_name, df in df_collection.items():
    print(f'\nProcessing {file_name}...')

    if 'N' in df.columns:
        df = df.drop(columns=['N'])
        print("  Dropped 'N' column.")
    elif '\ufeffN' in df.columns:
        df = df.drop(columns=['\ufeffN'])
        print("  Dropped BOM 'N' column.")

    current_arabic_cols = [col for col in arabic_cols_to_drop if col in df.columns]
    if current_arabic_cols:
        df = df.drop(columns=current_arabic_cols)
        print(f'  Dropped Arabic columns {current_arabic_cols}.')

    if 'import-2024.csv' in file_name:
        df['year'] = 2024
        print("  Injected 'year' = 2024.")

    df.columns = df.columns.str.strip()
    df = df.rename(columns=column_name_mapping)

    if 'month' in df.columns:
        df['month'] = df['month'].astype(str).str.extract(r'(\w+)$', expand=False)

    for col in expected_columns:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[expected_columns]
    cleaned_dataframes.append(df)
    print('  Done.')

# Concatenate all years
mast_df = pd.concat(cleaned_dataframes, ignore_index=True)
mast_df['commodity_no'] = mast_df['commodity_no'].astype(str)

# Verify leading zeros survived the pipeline
mask = mast_df['commodity_no'].str.startswith('0', na=False)
print(f'\nLeading-zero rows in commodity_no: {mask.sum()}')
print(mast_df.loc[mask, 'commodity_no'].head(5).tolist())
print(f"String 'nan' in commodity_no: {(mast_df['commodity_no'] == 'nan').sum()}")

# Remove stray newline characters
for col in mast_df.select_dtypes(include='object').columns:
    mast_df[col] = mast_df[col].astype(str).str.replace('\n', '').str.replace('\r', '')

# Export
mast_df.to_csv('bahrain_imports_cleaned.csv', index=False)
print("\n✓ Exported to 'bahrain_imports_cleaned.csv'")

---
## 4. Verification

Confirming data integrity before handing off to Power BI.

### 4.1 Shape & Schema

In [ ]:
print(f'Shape: {mast_df.shape}')
display(mast_df.info())

### 4.2 Missing Values

In [ ]:
display(mast_df.isnull().sum())

### 4.3 Row Count — Original vs Cleaned

In [ ]:
for file_name, df_orig in df_collection.items():
    year = int(file_name.replace('import-', '').replace('.csv', ''))
    orig_rows = len(df_orig)
    clean_rows = mast_df[mast_df['year'] == year].shape[0]
    match = '✓' if orig_rows == clean_rows else '✗'
    print(f'Year {year}: {orig_rows} → {clean_rows}  {match}')

### 4.4 Month Values

In [ ]:
print(sorted(mast_df['month'].unique()))

### 4.5 UN Code ↔ Country Name Correspondence

In [ ]:
pairs = mast_df[['un_code', 'country_name']].drop_duplicates()

bad_codes = pairs.groupby('un_code')['country_name'].nunique()
bad_codes = bad_codes[bad_codes > 1]

bad_names = pairs.groupby('country_name')['un_code'].nunique()
bad_names = bad_names[bad_names > 1]

print('UN codes → multiple countries:', bad_codes.to_dict() if not bad_codes.empty else 'None')
print('Country names → multiple codes:', bad_names.to_dict() if not bad_names.empty else 'None')

if bad_codes.empty and bad_names.empty:
    print('✓ One-to-one relationship confirmed.')

### 4.6 Random Sample

In [ ]:
display(mast_df.sample(5, random_state=42))